# Nour — admin: index, evaluate, review feedback

Run this whenever the books (`backend/data/*.txt`) or the chunking settings change. No GPU needed (CPU runtime is fine): embeddings are computed by Cloudflare Workers AI (BGE-M3), the same model the site uses for questions.

**Before the first run**, add these in Colab **Secrets** (🔑 icon on the left, turn on *Notebook access* for each):

| Name | Where to find it |
|---|---|
| `SUPABASE_URL` | Supabase → Project Settings → API → Project URL |
| `SUPABASE_SERVICE_KEY` | Supabase → Project Settings → API Keys → **secret** key (or legacy `service_role`) |
| `CLOUDFLARE_ACCOUNT_ID` | Cloudflare dashboard → Workers & Pages → Account ID (right column) |
| `CLOUDFLARE_API_TOKEN` | Cloudflare → My Profile → API Tokens → Create → template **Workers AI** |
| `GROQ_API_KEY` | console.groq.com → API Keys (only needed for step 5) |

And run `backend/supabase/schema.sql`, then `backend/supabase/learning.sql`, once in the Supabase SQL Editor.

In [ ]:
# 1. Get the code (fresh copy each run)
!rm -rf nour_project && git clone -q --depth 1 https://github.com/bachar88/nour_project.git
%cd nour_project/backend
!pip install -q httpx python-dotenv

In [ ]:
# 2. Load the keys from Colab Secrets into environment variables (read-only: nothing is printed)
import os
from google.colab import userdata

for name in ["SUPABASE_URL", "SUPABASE_SERVICE_KEY", "CLOUDFLARE_ACCOUNT_ID", "CLOUDFLARE_API_TOKEN", "GROQ_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
        print("✓", name)
    except Exception:
        print("✗", name, "— missing (GROQ_API_KEY is only needed for step 5)")

In [ ]:
# 3. Index: chunk the books → BGE-M3 embeddings (Cloudflare) → Supabase
#    Skips if nothing changed. Add --force to rebuild, or --book akhdari for one book.
!python -m scripts.index_supabase

In [ ]:
# 4. Evaluate retrieval on the Salat test set (no Groq calls)
#    Look at the suggested MIN_COSINE_SCORE at the end.
!python -m scripts.evaluate

In [ ]:
# 5. (optional) The exam with real answers from Groq — about 2,000 tokens per question, ~4 minutes.
#    --save keeps the scores in Supabase (table eval_runs) so you can compare versions.
#    Change the label to describe what you changed since the last run.
!python -m scripts.evaluate --llm --save --label "describe this version"

## The learning loop

1. Users press 👎 under a wrong answer, say what's wrong and pick the correct sentence in the reader.
2. **Step 7** lists the new reports. Read each one.
3. **Step 8**: accept the good ones (that creates a *correction*: a worked example the model sees for similar questions) or reject them.
4. Copy the printed line into `backend/eval/golden.jsonl` and push: the mistake becomes a permanent test.
5. Re-run **step 5** to check the score went up and nothing else went down.

Corrections are only used by the site once `backend/app/learning.py` is written and `USE_CORRECTIONS=true` is set on Vercel.

In [ ]:
# 6. History of the evaluation runs (newest first, read-only)
import json, os, httpx
key = os.environ["SUPABASE_SERVICE_KEY"]
h = {"apikey": key, **({"Authorization": f"Bearer {key}"} if key.startswith("eyJ") else {})}
runs = httpx.get(os.environ["SUPABASE_URL"] + "/rest/v1/eval_runs", headers=h,
                 params={"select": "id,created_at,label,scores", "order": "created_at.desc", "limit": "10"}).json()
for r in runs:
    print(f"#{r['id']}  {r['created_at'][:16]}  {r['label'] or '':30s}  {json.dumps(r['scores'], ensure_ascii=False)}")

In [ ]:
# 7. New feedback reports from the site (read-only)
!python -m scripts.review_feedback list

In [ ]:
# 8. Accept or reject a report — edit the id and the short correct answer, then run.
#    accept: evidence = the sentence the user picked (or add --evidence "…" to set it yourself)
#    It prints the line to add to backend/eval/golden.jsonl.
!python -m scripts.review_feedback accept 1 --answer "الجواب القصير الصحيح كما في الكتاب."
# !python -m scripts.review_feedback reject 2

## After evaluating

If step 4 suggests a different `MIN_COSINE_SCORE` (or you want to change `COSINE_MARGIN`, `MAX_K`…):
Vercel → your project → **Settings → Environment Variables** → add/edit the variable → **Deployments → Redeploy**.

- *Hit* `NO` for an in-scope question → the right text is not reaching the model: raise `COSINE_MARGIN` a little (e.g. 0.08) or `MAX_K` to 4.
- Out-of-scope questions answered → raise `MIN_COSINE_SCORE`.
- In-scope questions refused («لم يرد في النص…» with `abstained=retrieval`) → lower `MIN_COSINE_SCORE`.